# Exploratory Data Analysis (EDA)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)

This reduced notebook consolidates source notebooks `01_data_audit.ipynb` through
`05_dataset_builder.ipynb` from the `notebooks` branch. Large repeated diagnostics
and saved plots were removed. Values labeled **recorded snapshot** come from the
executed source notebooks and should be regenerated when the dataset changes.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# Point BTC_DATA_PATH to the canonical candles file. CSV and Parquet are supported.
DATA_PATH = Path(os.environ.get(
    "BTC_DATA_PATH",
    "data/processed/candles/BTCUSDT_15m_candles.parquet",
))

def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)
    elif path.suffix.lower() in {".csv", ".gz"}:
        frame = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported data format: {path.suffix}")
    for column in ("open_time", "close_time", "ingested_at"):
        if column in frame:
            frame[column] = pd.to_datetime(frame[column], utc=True, errors="coerce")
    if "open_time" in frame:
        frame = frame.sort_values("open_time").reset_index(drop=True)
    return frame

if DATA_PATH.is_file():
    df = load_table(DATA_PATH)
    print(f"Loaded {len(df):,} rows from {DATA_PATH}")
else:
    df = None
    print(
        "Dataset not bundled. Set BTC_DATA_PATH to the canonical BTCUSDT "
        "15-minute CSV/Parquet file to rerun the live checks."
    )


## Dataset Overview

**Source.** Public BTCUSDT spot-market candle data from
[Binance Data Collection](https://data.binance.vision/), supplemented by the
Binance REST API for the newest completed candles. Each observation is one UTC
15-minute OHLCV candle.

**Recorded snapshot (2020-01-01 00:00 UTC to 2026-07-31 20:15 UTC):**

| Characteristic | Value |
|---|---:|
| Raw observations | 230,618 |
| Raw columns | 15 |
| Unique candles | 230,618 |
| Observed calendar days | 2,404 |
| Source rows: Binance Vision / existing / REST | 227,656 / 2,445 / 517 |
| Valid engineered-feature rows | 229,434 |
| Leakage-safe modeling rows | 227,898 |
| Model features | 335 |
| Primary target | `target_binary_4` |
| Target meaning | 1 if the next one-hour log return exceeds 0.1%; otherwise 0 |

Raw columns are identifiers and timestamps; OHLC prices; base/quote volume;
trade count; taker-buy volume; source; and ingestion time. The 335 engineered
features cover trend (98), volume/trades (69), lags (50), volatility (41),
returns/momentum (28), regimes (19), candle/gap structure (15), calendar (13),
and two other controls. A 672-candle warm-up is removed before modeling.


In [ ]:
if df is not None:
    overview = pd.Series({
        "samples": len(df),
        "columns": df.shape[1],
        "start": df["open_time"].min() if "open_time" in df else None,
        "end": df["open_time"].max() if "open_time" in df else None,
        "duplicate_rows": int(df.duplicated().sum()),
    }, name="value")
    display(overview.to_frame())
    display(df.head(3))


## Handling Missing Values

The recorded raw snapshot contains **no null values**, no duplicate keys, no
conflicting duplicates, and no invalid OHLC rows. Therefore, values are not
mean-filled or forward-filled: doing so would fabricate market observations.

Time continuity is treated separately from cell-level missingness. The audit found
152 absent candles in 15 gap blocks (largest gap: 23 candles), corresponding to
99.9341% completeness over the observed range. It also flagged 13 zero-volume/
zero-trade candles and seven non-standard close-time rows. These warnings remain
visible; the feature and target builders validate alignment and drop incomplete
warm-up/future windows instead of silently imputing them.


In [ ]:
if df is not None:
    missing = df.isna().sum().rename("missing_count").to_frame()
    missing["missing_ratio"] = missing["missing_count"] / len(df)
    display(missing.query("missing_count > 0"))

    key = [c for c in ("symbol", "interval", "open_time") if c in df]
    print("Duplicate keys:", int(df.duplicated(key).sum()) if key else "n/a")

    if "open_time" in df:
        delta = df["open_time"].diff()
        gap_table = df.loc[delta.gt(pd.Timedelta(minutes=15)), ["open_time"]].copy()
        gap_table["gap"] = delta[delta.gt(pd.Timedelta(minutes=15))]
        gap_table["missing_candles"] = (
            gap_table["gap"] / pd.Timedelta(minutes=15) - 1
        ).astype(int)
        display(gap_table.head(20))


## Feature Distributions

The close price rose from 7,180.97 to 63,005.52 in the recorded window, with a
minimum of 3,882.22 and maximum of 126,011.18. A log price axis is therefore more
informative than a linear-only plot.

Fifteen-minute log returns are centered near zero (mean 0.000009, standard deviation
0.003380) but are strongly heavy-tailed: minimum -12.69%, maximum +20.40%, and
kurtosis 114.80. Volume is right-skewed (skewness 6.02) and positively related to
absolute returns (Spearman correlation 0.401). Absolute returns exhibit volatility
clustering: lag-1 autocorrelation is 0.376, while raw-return autocorrelation is weak.

For the one-hour multiclass view, UP/DOWN/NEUTRAL proportions are
38.35%/36.88%/24.77%. The binary modeling target has 87,307 positives (38.31%) and
140,591 negatives (61.69%).


In [ ]:
if df is not None and {"close", "volume"}.issubset(df.columns):
    plot = df.copy()
    plot["log_return_1"] = np.log(plot["close"]).diff()

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    plot.plot(x="open_time", y="close", logy=True, ax=axes[0, 0], legend=False)
    axes[0, 0].set_title("BTCUSDT close (log scale)")
    sns.histplot(plot["log_return_1"].dropna(), bins=100, ax=axes[0, 1])
    axes[0, 1].set_title("15-minute log returns")
    sns.histplot(np.log1p(plot["volume"]), bins=100, ax=axes[1, 0])
    axes[1, 0].set_title("log(1 + volume)")

    if "target_binary_4" in plot:
        sns.countplot(data=plot, x="target_binary_4", ax=axes[1, 1])
        axes[1, 1].set_title("One-hour binary target")
    else:
        axes[1, 1].axis("off")
    plt.tight_layout()
    plt.show()

    display(plot[["log_return_1", "volume"]].describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    ).T)


## Possible Biases

- **Class imbalance:** the positive class is 38.31% overall and varies by year
  (33.42% in 2023 versus 44.02% in 2021). Accuracy alone would reward a trivial
  negative predictor, so log loss, balanced accuracy, PR AUC, and calibration are
  reported.
- **Historical/regime shift:** volatility, volume, and target prevalence change over
  time. Random shuffling would leak future regimes into training; all splits are
  chronological.
- **Selection bias:** the study covers only BTCUSDT spot and one 15-minute venue feed.
  It does not establish general performance for other assets, exchanges, futures, or
  market microstructure data.
- **Measurement bias:** missing candles, zero-activity rows, exchange outages, and
  simplified transaction costs may differ from live execution.
- **Look-ahead risk:** rolling features use current/past information only. Targets,
  target end-times, and future-derived columns are excluded from `X`; purging and a
  96-candle embargo separate training and evaluation windows.


In [ ]:
if df is not None and {"open_time", "target_binary_4"}.issubset(df.columns):
    bias = df.assign(year=df["open_time"].dt.year).groupby("year")["target_binary_4"].agg(
        observations="size", positive_ratio="mean"
    )
    display(bias)
    bias["positive_ratio"].plot(marker="o", ylim=(0, 1), figsize=(9, 4))
    plt.axhline(df["target_binary_4"].mean(), color="black", ls="--", label="overall")
    plt.title("Target prevalence by year")
    plt.ylabel("positive ratio")
    plt.legend()
    plt.show()


## Correlations

Feature-to-target relationships are weak, which motivates regularized/nonlinear
models and strict out-of-time validation. In the recorded modeling table the largest
selected absolute Pearson correlations with `target_binary_4` were RSI-14 (-0.097),
16-candle historical volatility (+0.065), volatility regime (+0.053), four-candle
return (-0.050), and 96-candle historical volatility (+0.044).

Some feature pairs are redundant: volume versus taker-buy volume (0.994), adjacent
rolling-volume windows (up to 0.974), and one-candle return versus relative candle
body (1.000 after rounding). Tree regularization and linear-model scaling help, but
correlated features make individual importance values unstable and should not be
interpreted causally.


In [ ]:
if df is not None:
    candidates = [
        "target_binary_4", "rsi_14", "volatility_historical_16",
        "regime_volatility_code", "return_log_4",
        "volatility_historical_96", "macd_histogram_relative",
        "volume_zscore_96", "trade_count_zscore_96",
        "regime_trend_code", "return_log_1", "taker_imbalance",
    ]
    available = [column for column in candidates if column in df]
    if len(available) >= 2:
        corr = df[available].corr(numeric_only=True)
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1)
        plt.title("Selected feature and target correlations")
        plt.tight_layout()
        plt.show()

        if "target_binary_4" in corr:
            display(corr["target_binary_4"].drop("target_binary_4").abs()
                    .sort_values(ascending=False).rename("absolute_correlation"))


### EDA conclusion

The data passes the technical audit and is suitable for modeling after warm-up and
future-window removal. The main challenges are weak signal, heavy tails, regime drift,
moderate class imbalance, correlated engineered features, and realistic execution
costs—not ordinary missing-value cleanup. These findings determine the chronological,
probability-focused evaluation used in the next notebooks.
